In [ ]:
from urllib.parse import parse_qs, unquote, urlparse

import ipywidgets as widgets
from IPython.display import Javascript, Markdown, display

CURRENT_NOTEBOOK_URL = None
CURRENT_NOTEBOOK_QUERY = None
CURRENT_DEMO_PARAMS = {}


def _flatten_params(values):
    return {key: value[-1] if isinstance(value, list) else value for key, value in values.items()}


def _extract_params(current_url):
    parsed = urlparse(current_url)
    top_level = _flatten_params(parse_qs(parsed.query))

    nested = {}
    encoded_urlpath = top_level.get("urlpath", "")
    if encoded_urlpath:
        decoded_urlpath = unquote(encoded_urlpath)
        if "?" in decoded_urlpath:
            nested = _flatten_params(parse_qs(decoded_urlpath.split("?", 1)[1]))

    merged = {**top_level, **nested}
    merged.pop("urlpath", None)
    return parsed.query, top_level, nested, merged


url_widget = widgets.Text(description="URL:")
url_widget.add_class("demo-url-value")

search_widget = widgets.Text(description="Query:")
search_widget.add_class("demo-query-value")

output = widgets.Output()


def _refresh(_=None):
    global CURRENT_NOTEBOOK_URL, CURRENT_NOTEBOOK_QUERY, CURRENT_DEMO_PARAMS

    if not url_widget.value:
        return

    CURRENT_NOTEBOOK_URL = url_widget.value
    query, top_level, nested, merged = _extract_params(CURRENT_NOTEBOOK_URL)
    CURRENT_NOTEBOOK_QUERY = query
    CURRENT_DEMO_PARAMS = merged

    with output:
        output.clear_output(wait=True)
        display(Markdown("### Current URL"))
        print(CURRENT_NOTEBOOK_URL)
        print()
        display(Markdown("### Parsed Params"))
        print("Top-level query params:", top_level)
        print("Nested params from decoded urlpath:", nested)
        print("Merged params:", CURRENT_DEMO_PARAMS)
        print()
        print("Examples:")
        print("environments ->", CURRENT_DEMO_PARAMS.get("environments"))
        print("robots ->", CURRENT_DEMO_PARAMS.get("robots"))
        print("tasks ->", CURRENT_DEMO_PARAMS.get("tasks"))


url_widget.observe(_refresh, names="value")

display(widgets.VBox([url_widget, search_widget, output]))

display(Javascript("""
(() => {
  const setValue = (selector, value) => {
    const element = document.querySelector(selector);
    if (!element) {
      return false;
    }
    element.value = value;
    element.dispatchEvent(new Event('input', { bubbles: true }));
    element.dispatchEvent(new Event('change', { bubbles: true }));
    return true;
  };

  const updateWidgets = () => {
    const href = window.location.href;
    const search = window.location.search;
    const ready =
      setValue('.demo-url-value input', href) &&
      setValue('.demo-query-value input', search);

    if (!ready) {
      window.setTimeout(updateWidgets, 250);
    }
  };

  updateWidgets();
})();
"""))

print("Run this cell, then read CURRENT_NOTEBOOK_URL and CURRENT_DEMO_PARAMS in Python.")
